## Imports

In [1]:
import wandb
import logging
from tqdm import tqdm
from wandb.sdk.wandb_run import Run
import numpy as np
import pandas as pd
import plotly.express as px
import numpy as np
import plotly.graph_objs as go
import seaborn as sns
import matplotlib.pyplot as plt
from nn_core.common import PROJECT_ROOT
import json

/leonardo_work/IscrC_SLEY/agargiul/mass/.venv/lib/python3.11/site-packages/lightning_utilities/core/imports.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/leonardo_work/IscrC_SLEY/agargiul/mass/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [ ]:
from mass.utils.plots import Palette

plt.rcParams.update(
    {
        "text.usetex": True,
        "font.family": "serif",
        "axes.titlesize": 24,  # Larger axes/title fonts
        "axes.labelsize": 24,
        "xtick.labelsize": 24,
        "ytick.labelsize": 20,
        "legend.fontsize": 24,
    }
)
sns.set_context("talk")

cmap_name = "coolwarm_r"

palette = Palette(
    f"{PROJECT_ROOT}/misc/palette.json", map_path=f"{PROJECT_ROOT}/misc/palette_map.json"
)
palette

Project not installed in the current env, activate the correct env or install it with:
	pip install -e .


{'blue': '#335c67',
 'white': '#fff3b0',
 'yellow': '#e09f3e',
 'red': '#9e2a2b',
 'dark red': '#540b0e',
 'green': '#81b29a'}

## Get runs

In [30]:
api = wandb.Api()
entity, project = "gladia", "mass"  # set to your entity and project

In [31]:
def get_runs(entity, project, positive_tags, negative_tags):
    filters_pos_tags = {"$and": [{"tags": {"$eq": pos_tag}} for pos_tag in positive_tags]}
    filters_neg_tags = {}

    print(filters_pos_tags)
    filters = {**filters_pos_tags, **filters_neg_tags}
    runs = api.runs(entity + "/" + project, filters=filters)

    print(f"There are {len(runs)} runs respecting these conditions.")
    return runs

In [32]:
tags = ["TaskArithmetic"]

In [33]:
runs = get_runs(entity, project, positive_tags=tags, negative_tags=[])

{'$and': [{'tags': {'$eq': 'TaskArithmetic'}}]}
There are 9 runs respecting these conditions.


In [34]:
models = ["ViT-B-32", "ViT-B-16", "ViT-L-14"]

In [35]:
ref_run = runs[0]

In [36]:
print(set(ref_run.history().columns))

{'acc/test/EuroSAT', 'loss/test/RESISC45', 'normalized_acc/test/Flowers102', 'loss/test/EuroSAT', 'normalized_acc/test/MNIST', 'normalized_acc/test/FER2013', 'normalized_acc/test/Food101', 'normalized_acc/test/CIFAR100', 'loss/test/CIFAR10', 'loss/test/CIFAR100', 'acc/test/SVHN', 'normalized_acc/test/FashionMNIST', 'normalized_acc/test/CIFAR10', 'loss/test/Cars', 'loss/test/KMNIST', 'acc/test/FER2013', 'normalized_acc/test/SVHN', 'acc/test/Cars', 'normalized_acc/test/RESISC45', 'acc/test/Food101', 'acc/test/PCAM', 'normalized_acc/test/KMNIST', 'trainer/global_step', 'normalized_acc/test/EuroSAT', 'acc/test/STL10', '_timestamp', 'acc/test/Flowers102', 'acc/test/KMNIST', 'loss/test/FashionMNIST', 'loss/test/Flowers102', 'acc/test/RESISC45', 'loss/test/GTSRB', 'epoch', 'acc/test/OxfordIIITPet', 'normalized_acc/test/STL10', 'normalized_acc/test/GTSRB', 'normalized_acc/test/OxfordIIITPet', 'loss/test/PCAM', 'acc/test/CIFAR100', 'loss/test/SUN397', 'loss/test/SVHN', 'acc/test/CIFAR10', 'norm

In [37]:
print(ref_run.config["core/tags"])

['TaskArithmetic', 'benchmark', 'static_merge', 'n20', 'ViT-B-16']


#### Hparams

In [38]:
benchmarks = ["n8", "n14", "n20"]
models = ["ViT-B-32", "ViT-B-16", "ViT-L-14"]

In [39]:
avg_accs = {
    model: {benchmark: {"avg_acc": 0.0, "norm_acc": 0.0} for benchmark in benchmarks}
    for model in models
}

for run in runs:
    model = run.config["nn/encoder/model_name"]

    try:
        N = run.config["num_tasks"]
    except KeyError:
        N = run.config["ntasks"]

    benchmark = f"n{N}"

    avg_accs[model][benchmark]["avg_acc"] = run.summary["acc/test/avg"]
    avg_accs[model][benchmark]["norm_acc"] = run.summary["normalized_acc/test/avg"]

In [40]:
avg_accs

{'ViT-B-32': {'n8': {'avg_acc': 0.6882890909910202,
   'norm_acc': 0.7566415891051292},
  'n14': {'avg_acc': 0.64632448554039, 'norm_acc': 0.7250497852052961},
  'n20': {'avg_acc': 0.6402571007609368, 'norm_acc': 0.7190553784370423}},
 'ViT-B-16': {'n8': {'avg_acc': 0.7295745387673378,
   'norm_acc': 0.7833131328225136},
  'n14': {'avg_acc': 0.7064262947865895, 'norm_acc': 0.7702517126287732},
  'n20': {'avg_acc': 0.6900940157473088, 'norm_acc': 0.750011856853962}},
 'ViT-L-14': {'n8': {'avg_acc': 0.8437823727726936,
   'norm_acc': 0.8927241787314415},
  'n14': {'avg_acc': 0.8037028057234628, 'norm_acc': 0.8580883060182843},
  'n20': {'avg_acc': 0.7685202337801457, 'norm_acc': 0.8167705416679383}}}

In [41]:
# print latex

row = "&"
for model in models:

    for benchmark in benchmarks:
        avg_acc = avg_accs[model][benchmark]["avg_acc"]
        norm_acc = avg_accs[model][benchmark]["norm_acc"]

        row += f"${avg_acc*100:.1f}_{{({norm_acc*100:.1f})}}$ &"
print(row[:-1] + "\\\\")

&$68.8_{(75.7)}$ &$64.6_{(72.5)}$ &$64.0_{(71.9)}$ &$73.0_{(78.3)}$ &$70.6_{(77.0)}$ &$69.0_{(75.0)}$ &$84.4_{(89.3)}$ &$80.4_{(85.8)}$ &$76.9_{(81.7)}$ \\
